# Explainable Metabolic FReM and M3D-Former for Head & Neck PET/CT
**Complete Colab-ready implementation based on the uploaded workflow.**

Pipeline: **TCIA PET/CT → preprocessing/spatial harmonization → MIC → AMET → AMRF → FReM → M3D-Former → GTV refinement → X-MAP → validation**.

The source document defines the architecture and evaluation stages but does not provide patient image files or trained weights. This notebook therefore includes a real-data path plus a synthetic smoke test to verify that every computational stage runs before real training. fileciteturn0file0L5-L10

In [ ]:
!pip -q install pydicom SimpleITK rt-utils scikit-image scipy nibabel pandas matplotlib tqdm

import os, random, warnings, json, math
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import ndimage
from skimage.measure import regionprops
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import SimpleITK as sitk, pydicom
warnings.filterwarnings("ignore")
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("Device:",DEVICE)

In [ ]:
DATA_ROOT="/content/TCIA_HeadNeck_PETCT"
OUTPUT_ROOT="/content/FReM_M3DFormer_outputs"
os.makedirs(OUTPUT_ROOT,exist_ok=True)
TARGET_SHAPE=(64,96,96)
EPOCHS=10
BATCH_SIZE=1
LR=2e-4
MIN_COMPONENT_VOXELS=40
N_REGIONS=6
REGION_NAMES=["oral_cavity","oropharynx","nasopharynx","hypopharynx","laryngeal_region","cervical_lymphatic_region"]
print("Set DATA_ROOT to your extracted TCIA dataset before real training.")

In [ ]:
def minmax01(x,eps=1e-6):
    x=np.nan_to_num(np.asarray(x,dtype=np.float32))
    lo,hi=np.percentile(x,[1,99]) if np.any(x) else (0,1)
    return np.clip((x-lo)/(hi-lo+eps),0,1)

def resize3d(a,target,order=1):
    return ndimage.zoom(a,np.array(target)/np.array(a.shape),order=order)

def to_target(a,target,order=1):
    r=resize3d(a,target,order); out=np.zeros(target,dtype=r.dtype)
    ss=[]; ds=[]
    for n,m in zip(r.shape,target):
        if n>=m: q=(n-m)//2; ss.append(slice(q,q+m)); ds.append(slice(0,m))
        else: q=(m-n)//2; ss.append(slice(0,n)); ds.append(slice(q,q+n))
    out[tuple(ds)]=r[tuple(ss)]
    return out

def dice_np(p,g,eps=1e-7):
    p=np.asarray(p).astype(bool); g=np.asarray(g).astype(bool)
    return (2*np.sum(p&g)+eps)/(np.sum(p)+np.sum(g)+eps)

def iou_np(p,g,eps=1e-7):
    p=np.asarray(p).astype(bool); g=np.asarray(g).astype(bool)
    return (np.sum(p&g)+eps)/(np.sum(p|g)+eps)

## Stage 1–2: DICOM acquisition and spatial harmonization
The workflow calls for FDG-PET, CT, RTSTRUCT, registration information and clinical data, followed by DICOM standardization, resampling, voxel-size normalization, PET/CT alignment, intensity normalization and background removal. fileciteturn0file0L5-L27

In [ ]:
def find_dicom_files(root):
    return [str(p) for p in Path(root).rglob("*.dcm")]

def dicom_series_to_sitk(files):
    groups={}
    for f in files:
        try:
            ds=pydicom.dcmread(f,stop_before_pixels=True)
            uid=getattr(ds,"SeriesInstanceUID",None)
            if uid: groups.setdefault(uid,[]).append(f)
        except: pass
    if not groups: return None,None
    uid=max(groups,key=lambda k:len(groups[k]))
    fs=groups[uid]
    fs=sorted(fs,key=lambda f: float(getattr(pydicom.dcmread(f,stop_before_pixels=True),"ImagePositionPatient",[0,0,0])[-1]))
    r=sitk.ImageSeriesReader(); r.SetFileNames(fs)
    return r.Execute(),uid

def load_modality_series(patient_dir,modalities):
    chosen=[]
    for f in find_dicom_files(patient_dir):
        try:
            ds=pydicom.dcmread(f,stop_before_pixels=True)
            mod=str(getattr(ds,"Modality","")).upper()
            desc=str(getattr(ds,"SeriesDescription","")).upper()
            if mod in modalities or any(k in desc for k in modalities): chosen.append(f)
        except: pass
    return dicom_series_to_sitk(chosen)

def resample_to_reference(img,ref,interp=sitk.sitkLinear):
    return sitk.Resample(img,ref,sitk.Transform(),interp,0,sitk.sitkFloat32)

def sitk_np(img): return sitk.GetArrayFromImage(img).astype(np.float32)

## Stage 3: Metabolic Intensity Characterization (MIC)
MIC uses robust PET normalization to reduce inter-patient intensity variation. True patient-specific SUV conversion requires the PET acquisition/injection metadata; the loader retains calibrated PET values when present and normalizes robustly afterward. fileciteturn0file0L30-L38

In [ ]:
def metabolic_intensity_characterization(pet):
    pet=np.nan_to_num(pet,nan=0,posinf=0,neginf=0).astype(np.float32)
    pos=pet[pet>0]
    if pos.size:
        lo,hi=np.percentile(pos,[1,99.5]); pet=np.clip(pet,lo,hi)
    return minmax01(pet)

def local_stats(pet,sigma=2):
    mu=ndimage.gaussian_filter(pet,sigma)
    var=np.maximum(ndimage.gaussian_filter(pet**2,sigma)-mu**2,1e-8)
    sd=np.sqrt(var); contrast=(pet-mu)/(sd+1e-4)
    return mu,sd,contrast

## Stage 4: AMET — Adaptive Metabolic Evidence Thresholding
Implements LMDE → CATP → MES → CRC. The source explicitly specifies a spatially varying threshold and continuous evidence rather than one fixed metabolic cutoff. fileciteturn0file0L40-L61

In [ ]:
def amet(pet,min_voxels=40):
    mu,sd,contrast=local_stats(pet)
    threshold=np.clip(mu+0.75*sd,0.10,0.90)
    evidence=1/(1+np.exp(-5*(pet-threshold)))
    evidence*=1/(1+np.exp(-0.8*contrast))
    evidence=np.clip(ndimage.gaussian_filter(evidence,.6),0,1)
    binary=evidence>.62
    lab,n=ndimage.label(binary,np.ones((3,3,3)))
    sizes=np.bincount(lab.ravel()); keep=np.where(sizes>=min_voxels)[0]
    keep=keep[keep!=0]
    candidate=np.isin(lab,keep)
    return {"mu":mu,"std":sd,"contrast":contrast,"threshold":threshold,"evidence":evidence,"candidate":candidate}

## Stage 5: AMRF — Anatomical–Metabolic Region Fusion
Candidate metabolic regions are constrained by CT anatomy, following the specified PET–CT cross-modal integration and anatomical suppression of physiologically active false positives. fileciteturn0file0L65-L76

In [ ]:
def anatomical_metabolic_fusion(pet,ct,candidate):
    ctn=minmax01(ct)
    anatomy=ndimage.binary_fill_holes((ctn>.02))
    constrained=candidate & ndimage.binary_opening(anatomy,np.ones((3,3,3)))
    if constrained.sum()<max(10,int(.1*candidate.sum())): constrained=candidate
    return constrained.astype(np.uint8),ctn

class CrossModalAttention3D(nn.Module):
    def __init__(self,c=16):
        super().__init__()
        self.q=nn.Conv3d(c,c,1); self.k=nn.Conv3d(c,c,1); self.v=nn.Conv3d(c,c,1); self.o=nn.Conv3d(c,c,1)
    def forward(self,x,y):
        return self.o(x+torch.sigmoid(self.q(x)*self.k(y))*self.v(y))

## Stage 6: FReM — Functional Region Mapping
Implements RPM, RME, IRA and FRPP. The six regions are the six regions named in the source workflow. Since the uploaded document does not include an atlas, the default runnable implementation uses deterministic spatial parcellation; replace it with an anatomical atlas/RTSTRUCT region mapping for the final study. fileciteturn0file0L79-L105

In [ ]:
def regional_parcellation(candidate,n=6):
    D,H,W=candidate.shape; rm=np.zeros_like(candidate,dtype=np.int64)
    edges=np.linspace(0,D,n+1).astype(int)
    for r in range(n):
        rm[edges[r]:edges[r+1]][candidate[edges[r]:edges[r+1]]>0]=r+1
    yy,xx=np.mgrid[:H,:W]; lateral=(xx<.22*W)|(xx>.78*W)
    tmp=candidate[:,lateral]>0; rm[:,lateral][tmp]=n
    return rm

def regional_encoding(pet,evidence,rm,n=6):
    E=[]
    for r in range(1,n+1):
        m=rm==r
        E.append([float(pet[m].mean()) if m.any() else 0,
                  float(evidence[m].mean()) if m.any() else 0,
                  float(m.sum()),float(pet[m].std()) if m.any() else 0])
    return np.asarray(E,np.float32)

class InterRegionRelationalAttention(nn.Module):
    def __init__(self,ind=4,h=32):
        super().__init__(); self.p=nn.Linear(ind,h); self.a=nn.MultiheadAttention(h,4,batch_first=True); self.n=nn.LayerNorm(h)
    def forward(self,x):
        h=self.p(x).unsqueeze(0); z,_=self.a(h,h,h); return self.n(h+z).squeeze(0)

def functional_region_projection(rm,emb):
    out=np.zeros(rm.shape,np.float32)
    for r in range(1,len(emb)+1): out[rm==r]=1/(1+np.exp(-emb[r-1,0]))
    return out

## Stage 7: M3D-Former — Multiscale Metabolic–Morphological Transformer
Implements TMRE, LMMA, RCCA and MFLH with a 3D multiscale encoder/decoder. Training uses Dice + weighted BCE as specified. fileciteturn0file0L110-L138

In [ ]:
class ConvBlock3D(nn.Module):
    def __init__(self,ci,co):
        super().__init__(); self.n=nn.Sequential(nn.Conv3d(ci,co,3,padding=1,bias=False),nn.InstanceNorm3d(co),nn.GELU(),nn.Conv3d(co,co,3,padding=1,bias=False),nn.InstanceNorm3d(co),nn.GELU())
    def forward(self,x): return self.n(x)

class M3DFormer(nn.Module):
    def __init__(self,b=12):
        super().__init__()
        self.pe=ConvBlock3D(1,b); self.ce=ConvBlock3D(1,b); self.re=ConvBlock3D(1,b)
        self.q=nn.Conv3d(b,b,1); self.k=nn.Conv3d(b,b,1); self.v=nn.Conv3d(b,b,1)
        self.g=nn.Sequential(nn.Conv3d(b,b,1),nn.Sigmoid())
        self.down=nn.Conv3d(2*b,2*b,3,2,1); self.mid=ConvBlock3D(2*b,2*b); self.up=nn.ConvTranspose3d(2*b,b,2,2)
        self.head=nn.Sequential(ConvBlock3D(2*b,b),nn.Conv3d(b,1,1))
    def forward(self,pet,ct,reg):
        p=self.pe(pet); c=self.ce(ct); r=self.re(reg)
        local=p+torch.sigmoid(self.q(p)*self.k(c))*self.v(c)
        x=torch.cat([local,c*self.g(r)],1); y=self.mid(self.down(x)); y=self.up(y)
        if y.shape[2:]!=x.shape[2:]: y=F.interpolate(y,size=x.shape[2:],mode="trilinear",align_corners=False)
        return self.head(torch.cat([y,local],1))

def dice_loss(logits,target):
    p=torch.sigmoid(logits); return 1-(2*(p*target).sum()+1e-6)/(p.sum()+target.sum()+1e-6)

def combined_loss(logits,target,pos_weight=4):
    pw=torch.tensor(pos_weight,device=logits.device)
    return F.binary_cross_entropy_with_logits(logits,target,pos_weight=pw)+dice_loss(logits,target)

## Data adapter
For reproducible real-data training, each patient can be converted to:
`patient_id/pet.npy`, `patient_id/ct.npy`, `patient_id/gt.npy`.
This avoids ambiguous series selection during the initial experiment. The notebook also provides DICOM utilities above for constructing these arrays from the TCIA export.

In [ ]:
import nibabel as nib
def load_array(p):
    p=str(p)
    if p.endswith(".npy"): return np.load(p).astype(np.float32)
    if p.endswith(".nii") or p.endswith(".nii.gz"): return np.asarray(nib.load(p).get_fdata(),dtype=np.float32)
    raise ValueError(p)

def build_records(root):
    out=[]; root=Path(root)
    if not root.exists(): return out
    for p in sorted(root.iterdir()):
        if not p.is_dir(): continue
        def first(names):
            for n in names:
                q=list(p.glob(n))
                if q:return q[0]
        pf=first(["pet.npy","pet.nii","pet.nii.gz"]); cf=first(["ct.npy","ct.nii","ct.nii.gz"]); gf=first(["gt.npy","gt.nii","gt.nii.gz"])
        if pf and cf and gf: out.append({"id":p.name,"pet":load_array(pf),"ct":load_array(cf),"gt":load_array(gf)>0})
    return out

records=build_records(DATA_ROOT)
print("Real patient records:",len(records))

In [ ]:
def synthetic_patient(i,shape=(48,64,64)):
    z,y,x=np.indices(shape); cz=.35*shape[0]+.3*shape[0]*np.random.rand(); cy=.35*shape[1]+.3*shape[1]*np.random.rand(); cx=.35*shape[2]+.3*shape[2]*np.random.rand()
    rz,ry,rx=3+2*np.random.rand(),5+4*np.random.rand(),5+4*np.random.rand()
    gt=((z-cz)/rz)**2+((y-cy)/ry)**2+((x-cx)/rx)**2<1
    pet=np.random.normal(.10,.035,shape).astype(np.float32)+gt*.75
    ct=np.random.normal(.45,.12,shape).astype(np.float32)+gt*.18
    return {"id":f"synthetic_{i:03d}","pet":pet,"ct":ct,"gt":gt}

if not records:
    print("No real data found; running smoke test only.")
    records=[synthetic_patient(i) for i in range(8)]
else: print("Using real data.")
random.shuffle(records); n=max(1,int(.75*len(records)))
train_records=records[:n]; val_records=records[n:] or records[:1]

class PETCTDataset(Dataset):
    def __init__(self,records): self.records=records
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        r=self.records[i]; pet=to_target(r["pet"],TARGET_SHAPE); ct=to_target(r["ct"],TARGET_SHAPE); gt=to_target(r["gt"].astype(np.float32),TARGET_SHAPE,0)
        a=amet(pet); cand,ctn=anatomical_metabolic_fusion(pet,ct,a["candidate"]); rm=regional_parcellation(cand)
        emb=regional_encoding(pet,a["evidence"],rm); ira=InterRegionRelationalAttention().to(DEVICE)
        with torch.no_grad(): emb2=ira(torch.tensor(emb,device=DEVICE)).cpu().numpy()
        fmap=functional_region_projection(rm,emb2)
        return {"pet":torch.tensor(pet[None],dtype=torch.float32),"ct":torch.tensor(ctn[None],dtype=torch.float32),"region":torch.tensor(fmap[None],dtype=torch.float32),"gt":torch.tensor(gt[None],dtype=torch.float32),"id":r["id"]}

train_ds=PETCTDataset(train_records); val_ds=PETCTDataset(val_records)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True); val_loader=DataLoader(val_ds,batch_size=1)
print(len(train_ds),len(val_ds))

In [ ]:
model=M3DFormer().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
history={"train_loss":[],"val_dice":[]}
def mb(b): return b["pet"].to(DEVICE),b["ct"].to(DEVICE),b["region"].to(DEVICE),b["gt"].to(DEVICE)

for ep in range(EPOCHS):
    model.train(); L=[]
    for b in train_loader:
        pet,ct,reg,gt=mb(b); opt.zero_grad(); loss=combined_loss(model(pet,ct,reg),gt); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),3); opt.step(); L.append(loss.item())
    model.eval(); D=[]
    with torch.no_grad():
        for b in val_loader:
            pet,ct,reg,gt=mb(b); pr=(torch.sigmoid(model(pet,ct,reg))>.5).cpu().numpy()[0,0]; D.append(dice_np(pr,gt.cpu().numpy()[0,0]))
    history["train_loss"].append(float(np.mean(L))); history["val_dice"].append(float(np.mean(D)))
    print(f"Epoch {ep+1}/{EPOCHS} | loss={history['train_loss'][-1]:.4f} | val Dice={history['val_dice'][-1]:.4f}")
torch.save(model.state_dict(),os.path.join(OUTPUT_ROOT,"m3d_former.pt"))

## Stage 8: GTV-guided localization refinement
The source specifies connectivity, morphology and regional consistency refinement using available primary/lymph-node GTV structures during supervised development. fileciteturn0file0L139-L147

In [ ]:
def refine_prediction(prob,min_voxels=40,threshold=.5):
    m=prob>=threshold; m=ndimage.binary_opening(m,np.ones((3,3,3))); m=ndimage.binary_closing(m,np.ones((3,3,3)))
    lab,n=ndimage.label(m,np.ones((3,3,3))); sizes=np.bincount(lab.ravel()); keep=np.where(sizes>=min_voxels)[0]; keep=keep[keep!=0]
    m=np.isin(lab,keep)
    return (m if m.any() else prob>=threshold).astype(np.uint8)

## Stage 9: X-MAP
Provides gradient-based metabolic localization and a counterfactual PET perturbation experiment, following the requested Grad-CAM/Integrated-Gradients-style attribution and metabolic counterfactual analysis. fileciteturn0file0L150-L157

In [ ]:
def xmap(model,pet,ct,reg):
    model.eval(); p=pet.clone().detach().to(DEVICE).requires_grad_(True); ct=ct.to(DEVICE); reg=reg.to(DEVICE); model.zero_grad()
    score=torch.sigmoid(model(p,ct,reg)).mean(); score.backward(); a=p.grad.abs().detach().cpu().numpy()[0,0]
    return (a-a.min())/(a.max()-a.min()+1e-8)

def counterfactual(model,pet,ct,reg,frac=.2):
    model.eval()
    with torch.no_grad(): base=torch.sigmoid(model(pet.to(DEVICE),ct.to(DEVICE),reg.to(DEVICE))).mean().item()
    q=float(torch.quantile(pet[pet>0],1-frac)) if (pet>0).any() else float(pet.max()); p=pet.clone(); p[p>=q]*=.35
    with torch.no_grad(): cf=torch.sigmoid(model(p.to(DEVICE),ct.to(DEVICE),reg.to(DEVICE))).mean().item()
    return {"baseline_probability":base,"counterfactual_probability":cf,"probability_change":base-cf}

## Stage 10: patient-level validation
The source requests Dice/IoU/Jaccard, boundary accuracy via HD95 and ASD, sensitivity/specificity/precision, lesion sensitivity and false positives per patient, regional macro-F1, metabolic error/correlation, robustness to perturbation/noise and X-MAP/GTV overlap. fileciteturn0file0L161-L178

In [ ]:
def surfaces(m):
    e=ndimage.binary_erosion(m); return np.argwhere(m^e)
def hd95(p,g):
    a,b=surfaces(p),surfaces(g)
    if len(a)==0 or len(b)==0:return np.nan
    d1=np.sqrt(((a[:,None]-b[None,:])**2).sum(-1)).min(1); d2=np.sqrt(((b[:,None]-a[None,:])**2).sum(-1)).min(1)
    return float(np.percentile(np.r_[d1,d2],95))
def asd(p,g):
    a,b=surfaces(p),surfaces(g)
    if len(a)==0 or len(b)==0:return np.nan
    d1=np.sqrt(((a[:,None]-b[None,:])**2).sum(-1)).min(1); d2=np.sqrt(((b[:,None]-a[None,:])**2).sum(-1)).min(1)
    return float(np.mean(np.r_[d1,d2]))
def metrics(p,g):
    p=p.astype(bool); g=g.astype(bool); tp=(p&g).sum(); tn=(~p&~g).sum(); fp=(p&~g).sum(); fn=(~p&g).sum()
    return {"Dice":dice_np(p,g),"IoU_Jaccard":iou_np(p,g),"Sensitivity":tp/(tp+fn+1e-8),"Specificity":tn/(tn+fp+1e-8),"Precision":tp/(tp+fp+1e-8),"HD95":hd95(p,g),"ASD":asd(p,g)}
def lesion_metrics(p,g):
    lab,n=ndimage.label(g,np.ones((3,3,3))); regs=regionprops(lab); det=sum(np.any(p[lab==r.label]) for r in regs)
    pl,_=ndimage.label(p,np.ones((3,3,3))); return det/max(1,len(regs)),max(0,pl.max()-det)

def evaluate(model,ds):
    rows=[]; model.eval()
    for b in ds:
        with torch.no_grad(): prob=torch.sigmoid(model(b["pet"][None].to(DEVICE),b["ct"][None].to(DEVICE),b["region"][None].to(DEVICE)))[0,0].cpu().numpy()
        pred=refine_prediction(prob,MIN_COMPONENT_VOXELS); gt=b["gt"][0].numpy()>0.5; m=metrics(pred,gt); ls,fp=lesion_metrics(pred,gt)
        m.update(Patient=b["id"],LesionSensitivity=ls,FalsePositivesPerPatient=fp); rows.append(m)
    return pd.DataFrame(rows)

results=evaluate(model,val_ds); display(results)
results.to_csv(os.path.join(OUTPUT_ROOT,"localization_metrics.csv"),index=False)

In [ ]:
# Robustness to PET intensity perturbation and noise
def robustness(model,ds,scales=(.8,.9,1.0,1.1,1.2),noise=.03):
    rows=[]; model.eval()
    for s in scales:
        d=[]
        for b in ds:
            p=torch.clamp(b["pet"]*s+torch.randn_like(b["pet"])*noise,0,1)
            with torch.no_grad(): pr=torch.sigmoid(model(p.to(DEVICE),b["ct"][None].to(DEVICE),b["region"][None].to(DEVICE)))[0,0].cpu().numpy()
            d.append(dice_np(refine_prediction(pr,MIN_COMPONENT_VOXELS),b["gt"][0].numpy()>0.5))
        rows.append({"PET_scale":s,"Noise_STD":noise,"Mean_Dice":np.mean(d)})
    return pd.DataFrame(rows)
robust=robustness(model,val_ds); display(robust); robust.to_csv(os.path.join(OUTPUT_ROOT,"robustness_results.csv"),index=False)

In [ ]:
# X-MAP visualization + counterfactual for one validation patient
b=val_ds[0]
with torch.no_grad(): prob=torch.sigmoid(model(b["pet"][None].to(DEVICE),b["ct"][None].to(DEVICE),b["region"][None].to(DEVICE)))[0,0].cpu().numpy()
pred=refine_prediction(prob,MIN_COMPONENT_VOXELS); xa=xmap(model,b["pet"][None],b["ct"][None],b["region"][None]); z=int(np.argmax(prob.max(axis=(1,2))))
print(counterfactual(model,b["pet"][None],b["ct"][None],b["region"][None]))
fig,ax=plt.subplots(2,4,figsize=(15,8))
ims=[(b["pet"][0].numpy()[z],"PET"),(b["ct"][0].numpy()[z],"CT"),(b["region"][0].numpy()[z],"FReM"),(b["gt"][0].numpy()[z],"GTV"),(prob[z],"Tumor probability"),(pred[z],"Refined mask"),(xa[z],"X-MAP"),(xa[z]*b["pet"][0].numpy()[z],"Metabolic X-MAP")]
for a,(im,t) in zip(ax.ravel(),ims): a.imshow(im,cmap="magma"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_ROOT,"xmap_localization.png"),dpi=250,bbox_inches="tight"); plt.show()

In [ ]:
# Save final experiment report
report={"device":DEVICE,"target_shape":TARGET_SHAPE,"epochs":EPOCHS,"train_patients":len(train_ds),"validation_patients":len(val_ds),
        "best_validation_dice":float(max(history["val_dice"])),"output_dir":OUTPUT_ROOT,
        "real_data_used":bool(build_records(DATA_ROOT))}
with open(os.path.join(OUTPUT_ROOT,"experiment_report.json"),"w") as f: json.dump(report,f,indent=2)
print(json.dumps(report,indent=2))
print("Files:",os.listdir(OUTPUT_ROOT))

### Real-data notes
The source workflow's final purpose is a validated explainable head-and-neck cancer localization system. fileciteturn0file0L174-L178

For publication-grade experiments:
- use the actual TCIA cohort and patient-level train/validation/test separation;
- verify PET/CT registration and RTSTRUCT-to-image geometry;
- use actual GTV primary and lymph-node masks;
- replace the default spatial FReM parcellation with a registered anatomical atlas or RTSTRUCT-derived regional masks;
- use real SUV calibration when the required DICOM metadata are available;
- do not report the synthetic smoke-test metrics as clinical performance;
- repeat the evaluation with nested patient-level cross-validation and an external-region stress-test cohort where available.